In [12]:
import gzip
from tqdm import tqdm
from collections import defaultdict

In [13]:
def readGz(path):
    for l in gzip.open(path, "rt"):
        yield eval(l)


def readCSV(path):
    f = gzip.open(path, "rt")
    f.readline()
    for l in f:
        yield l.strip().split(",")

In [14]:
allRatings = [(u, b, int(r)) for u, b, r in readCSV("train_Interactions.csv.gz")]
ratingsPerUser = defaultdict(list)
ratingsPerItem = defaultdict(list)
for u, b, r in allRatings:
    ratingsPerUser[u].append((b, r))
    ratingsPerItem[b].append((u, r))

In [ ]:
totalRead = 0
bookCount = defaultdict(int)
userCount = defaultdict(int)
for user,book,_ in allRatings:
    bookCount[book] += 1
    userCount[user] += 1
    totalRead += 1

In [16]:
def build_popular_books(bookCount, totalRead, threshold=0.72):
    mostPopular = [(bookCount[x], x) for x in bookCount]
    mostPopular.sort()
    mostPopular.reverse()
    
    count = 0
    popular = set()
    for ic, i in mostPopular:
        count += ic
        popular.add(i)
        if count > totalRead * threshold:
            break
    return popular

def build_active_users(userCount, totalRead, threshold=0.01):
    mostActive = [(userCount[x], x) for x in userCount]
    mostActive.sort()
    mostActive.reverse()
    
    count = 0
    active = set()
    for ic, i in mostActive:
        count += ic
        active.add(i)
        if count > totalRead * threshold:
            break
    return active

In [17]:
userBooks = {u: set(b for b, _ in bs) for u, bs in ratingsPerUser.items()}
bookUsers = {b: set(u for u, _ in us) for b, us in ratingsPerItem.items()}
popularBooks = build_popular_books(bookCount, totalRead)
activeUsers = build_active_users(userCount, totalRead)

def Jaccard(s1, s2):
    return len(set(s1) & set(s2)) / len(set(s1) | set(s2))

def predictRead(u, b):
    if u not in userBooks or not userBooks[u]:
        return int(b in popularBooks)

    if b not in bookUsers or not bookUsers[b]:
        return int(u in activeUsers)
    
    maxSim = 0
    for bp in userBooks[u]:
        if bp in bookUsers:
            sim = Jaccard(bookUsers[b], bookUsers[bp])
            maxSim = max(maxSim, sim)
            if maxSim > 0.025:
                return 1
    
    return int(b in popularBooks)

In [18]:
predictions = open("predictions_Read.csv", "w")
for l in tqdm(open("pairs_Read.csv"), desc="Writing read predictions"):
    if l.startswith("userID"):
        predictions.write(l)
        continue
    u, b = l.strip().split(",")
    predictions.write(f"{u},{b},{predictRead(u, b)}\n")
predictions.close()

Writing read predictions: 20001it [00:02, 6933.83it/s]
